In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import numpy.linalg as la
from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. GLOBAL CONFIGURATION & DATA LOADING
# ==============================================================================
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
master_file = PROCESSED_DIR / "master_panel_data.csv"

df = pd.read_csv(master_file)
df = df.set_index(['Country Name', 'Year']).sort_index()

BRICS_NATIONS = ["Brazil", "Russia", "India", "China", "Egypt", "United Arab Emirates"]
OECD_EMERGING = ["Chile", "Colombia", "Costa Rica", "Turkey", "Hungary", "Poland"]

def categorize_economy(country):
    if country in BRICS_NATIONS: return 'BRICS (Developing)'
    elif country in OECD_EMERGING: return 'OECD (Emerging)'
    else: return 'OECD (Advanced)'

df['Economic_Bloc'] = [categorize_economy(c) for c, y in df.index]

# Create Elite Capture Ratio
TOP_10_COL = 'Wealth Inequality (Top 10%)'
BOTTOM_50_COL = 'Wealth Inequality (Bottom 50%)'
df['Wealth_Ratio_Top10_to_Bottom50'] = df[TOP_10_COL] / df[BOTTOM_50_COL]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

TIME_SPLITS = {
    "PRE-CRISIS ERA (1995-2008)": (1995, 2008),
    "POST-CRISIS ERA (2009-2022)": (2009, 2022)
}

Y_VAR = 'Wealth_Ratio_Top10_to_Bottom50'
# Matched perfectly to 03_Baseline, with GDP Growth acting as a control instead of the target
X_VARS = [
    'General Government Debt (% of GDP)', 
    'GDP growth (annual %)',
    'Gross capital formation (% of GDP)',
    'Inflation (Annual %)',
    'Population growth (annual %)'
]

# ==============================================================================
# 2. EXCEL FORMATTER & ECONOMETRIC FUNCTIONS
# ==============================================================================
def print_excel_summary(res, model_title):
    """Parses linearmodels output into a strict Excel Solver format."""
    print(f"\nSUMMARY OUTPUT: {model_title}")
    
    nobs = res.nobs
    r2 = getattr(res, 'rsquared_inclusive', getattr(res, 'rsquared', np.nan))
    mult_r = np.sqrt(max(r2, 0)) if pd.notnull(r2) else "#N/A"
    
    df_resid = getattr(res, 'df_resid', np.nan)
    df_model = getattr(res, 'df_model', np.nan)
    
    adj_r2 = 1 - (1 - r2) * (nobs - 1) / df_resid if pd.notnull(df_resid) and df_resid > 0 else "#N/A"
    
    ssr = getattr(res, 'resid_ss', np.nan)
    sst = getattr(res, 'total_ss', np.nan)
    std_err = np.sqrt(ssr / df_resid) if pd.notnull(ssr) and df_resid > 0 else "#N/A"
    
    # Print Regression Statistics
    print("\nRegression Statistics")
    print(f"{'Multiple R':<20} {mult_r if isinstance(mult_r, str) else f'{mult_r:.8f}'}")
    print(f"{'R Square':<20} {r2 if isinstance(r2, str) else f'{r2:.8f}'}")
    print(f"{'Adjusted R Square':<20} {adj_r2 if isinstance(adj_r2, str) else f'{adj_r2:.8f}'}")
    print(f"{'Standard Error':<20} {std_err if isinstance(std_err, str) else f'{std_err:.8f}'}")
    print(f"{'Observations':<20} {nobs}")
    
    # Print ANOVA
    print("\nANOVA")
    print(f"{'':<12} {'df':<10} {'SS':<15} {'MS':<15} {'F':<15} {'Significance F'}")
    
    if pd.notnull(ssr) and pd.notnull(sst):
        ss_reg = sst - ssr
        ms_reg = ss_reg / df_model if df_model > 0 else np.nan
        ms_res = ssr / df_resid if df_resid > 0 else np.nan
        
        f_stat = getattr(res.f_statistic, 'stat', np.nan) if hasattr(res, 'f_statistic') else (ms_reg/ms_res if ms_res > 0 else np.nan)
        p_f = getattr(res.f_statistic, 'pval', np.nan) if hasattr(res, 'f_statistic') else np.nan
        
        print(f"{'Regression':<12} {df_model:<10} {ss_reg:<15.5E} {ms_reg:<15.5E} {f_stat:<15.6f} {p_f:.6f}")
        print(f"{'Residual':<12} {df_resid:<10} {ssr:<15.5E} {ms_res:<15.5E}")
        print(f"{'Total':<12} {nobs-1:<10} {sst:<15.5E}")
    else:
        print("ANOVA not applicable for this estimator.")
        
    # Print Coefficients
    print("\n" + "-"*120)
    print(f"{'':<40} {'Coefficients':<15} {'Standard Error':<15} {'t Stat':<10} {'P-value':<10} {'Lower 95%':<12} {'Upper 95%'}")
    
    params = res.params
    se = res.std_errors
    tstat = res.tstats
    pval = res.pvalues
    ci = res.conf_int()
    
    for var in params.index:
        c, s, t, p = params[var], se[var], tstat[var], pval[var]
        l, u = ci.loc[var, 'lower'], ci.loc[var, 'upper']
        print(f"{var[:38]:<40} {c:<15.6f} {s:<15.6f} {t:<10.6f} {p:<10.6f} {l:<12.6f} {u:<12.6f}")
    print("-" * 120 + "\n")


def hausman_test(fe_res, re_res):
    """Calculates the Hausman Test statistic to choose between FE and RE."""
    common_vars = set(fe_res.params.index).intersection(re_res.params.index)
    common_vars = [v for v in common_vars if v != 'const']
    b_fe, b_re = fe_res.params[common_vars], re_res.params[common_vars]
    v_fe, v_re = fe_res.cov.loc[common_vars, common_vars], re_res.cov.loc[common_vars, common_vars]
    df_haus = len(common_vars)
    b_diff, v_diff = b_fe - b_re, v_fe - v_re
    try: h_stat = float(b_diff.T @ la.inv(v_diff) @ b_diff)
    except la.LinAlgError: h_stat = float(b_diff.T @ la.pinv(v_diff) @ b_diff)
    return h_stat, stats.chi2.sf(h_stat, df_haus)


def execute_panel_models(data, dependent, independents, bloc_name):
    """Runs FE, RE, Hausman Test, and exact Excel Output."""
    reg_data = data[[dependent] + independents].dropna()
    if len(reg_data) < 30: return
    
    n_entities = reg_data.index.get_level_values(0).nunique()
    n_vars = len(independents) + 1 
    
    Y = reg_data[dependent]
    X = sm.add_constant(reg_data[independents])
    
    # 1. Fixed Effects Model
    fe_model = PanelOLS(Y, X, entity_effects=True, time_effects=True)
    fe_res = fe_model.fit(cov_type='robust')
    
    # 2. Random Effects Model & Hausman
    try:
        if n_entities <= n_vars + 1: raise ValueError("Small N")
        re_model = RandomEffects(Y, X)
        re_res = re_model.fit(cov_type='robust')
        h_stat, h_pval = hausman_test(fe_res, re_res)
        print(f"\n[HAUSMAN TEST] Chi2 = {h_stat:.2f} | p-value = {h_pval:.4f} -> {'Fixed Effects' if h_pval < 0.05 else 'Random Effects'} Preferred")
    except Exception:
        pass # Silently skip Hausman if the sample is too small
        
    # 3. Print Custom Excel Output
    print_excel_summary(fe_res, model_title=bloc_name)


# ==============================================================================
# 3. PIPELINE EXECUTION
# ==============================================================================
if __name__ == "__main__":
    for time_label, (start_yr, end_yr) in TIME_SPLITS.items():
        print("\n" + "#"*120)
        print(f"### {time_label} ###")
        print("#"*120)
        
        df_time = df.loc[(slice(None), slice(start_yr, end_yr)), :]
        
        # Global Panel
        execute_panel_models(df_time, Y_VAR, X_VARS, "GLOBAL PANEL (ALL ENTITIES)")
        
        # Bloc Panels
        for bloc in ['OECD (Advanced)', 'OECD (Emerging)', 'BRICS (Developing)']:
            df_bloc = df_time[df_time['Economic_Bloc'] == bloc]
            execute_panel_models(df_bloc, Y_VAR, X_VARS, bloc)
            


########################################################################################################################
### PRE-CRISIS ERA (1995-2008) ###
########################################################################################################################

[HAUSMAN TEST] Chi2 = 2.63 | p-value = 0.7563 -> Random Effects Preferred

SUMMARY OUTPUT: GLOBAL PANEL (ALL ENTITIES)

Regression Statistics
Multiple R           0.83647229
R Square             0.69968590
Adjusted R Square    0.66389089
Standard Error       24.77194096
Observations         555

ANOVA
             df         SS              MS              F               Significance F
Regression   60         3.31364E+03     5.52273E+01     1.079977        0.370509
Residual     495        3.03756E+05     6.13649E+02    
Total        554        3.07070E+05    

------------------------------------------------------------------------------------------------------------------------
                                